<a href="https://colab.research.google.com/github/A-ros1076/BUS118s/blob/Dev/Group_Exercise_Agentic_AI_in_Customer_Service_and_Sales.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dyson Customer Service Multi-Agent Chatbot
## BUS118s | AI and Business Applications

This notebook implements a Dyson customer service chatbot using:
- **OEPN API** Used because Gemini was rate limiting hard
- **LangGraph** `StateGraph` for multi-agent routing architecture
- **LangGraph `MemorySaver`** for conversation memory across turns

### Architecture
| Agent | Responsibility | Tools |
|---|---|---|
| **Router** | Classifies input → PRODUCT / ORDER / REFUND / SMALLTALK / END | None |
| **Product Agent** | Features, pricing, recommendations | `get_dyson_price()` |
| **Orders Agent** | Order lookup and quantity updates | `get_order_details()`, `update_quantity()` |
| **Refund Agent** | Return and refund policy questions | None (policy in system prompt) |

In [80]:
# ── Install Required Packages ─────────────────────────────────────────────────
# langchain-google-genai : Gemini integration for LangChain
# langgraph              : StateGraph, MemorySaver, create_react_agent
# langchain              : Core message types, tool decorator
# pandas                 : CSV loading for product/order data
!pip install langchain langchain-openai langgraph pandas -q

## Step 1 — API Setup & Model Initialization

In [81]:
from langchain_openai import ChatOpenAI
from google.colab import userdata

model = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=userdata.get("OPENAI_API_KEY")
)

## Step 2 — CSV Data Files

In [82]:
# ── Step 2: Create CSV Data Files ────────────────────────────────────────────
# Both CSVs are generated here so the notebook is fully self-contained in Colab.
import pandas as pd

# ── dyson_products.csv ────────────────────────────────────────────────────────
# Used by the Product Agent's get_dyson_price() tool
products_data = {
    "Name": [
        "V15 Detect Absolute",
        "V12 Detect Slim",
        "Cyclone V10",
        "Purifier Cool Formaldehyde",
        "Airwrap Multi-Styler",
        "Supersonic Hairdryer"
    ],
    "Price": [749, 649, 499, 649, 599, 429],
    "Category": ["Vacuum", "Vacuum", "Vacuum", "Air Purifier", "Hair Care", "Hair Care"],
    "Features": [
        "Laser dust detection, LCD screen, strongest suction",
        "Lightweight, laser detection, great for smaller homes",
        "Reliable cordless, strong suction, great value",
        "HEPA H13, removes formaldehyde, cooling fan",
        "Curls, dries, styles with Coanda effect, no extreme heat",
        "Fast drying, reduces heat damage, lightweight"
    ]
}
products_df = pd.DataFrame(products_data)
products_df.to_csv("dyson_products.csv", index=False)
print("dyson_products.csv:")
print(products_df.to_string(index=False))

print()

# ── dyson_orders.csv ─────────────────────────────────────────────────────────
# Used by the Orders Agent's get_order_details() and update_quantity() tools
orders_data = {
    "Order ID": ["DYS-483921", "DYS-291047", "DYS-837562", "DYS-104839", "DYS-667234"],
    "Product": [
        "V15 Detect Absolute",
        "Airwrap Multi-Styler",
        "Purifier Cool Formaldehyde",
        "Supersonic Hairdryer",
        "V12 Detect Slim"
    ],
    "Quantity Ordered": [1, 2, 1, 1, 3],
    "Order Date": ["2025-03-01", "2025-03-05", "2025-02-28", "2025-03-10", "2025-03-12"],
    "Delivery Date": ["2025-03-08", "2025-03-12", "2025-03-07", "2025-03-17", "2025-03-19"],
    "Status": ["Shipped", "Processing", "Delivered", "Shipped", "Processing"]
}
orders_df = pd.DataFrame(orders_data)
orders_df.to_csv("dyson_orders.csv", index=False)
print("dyson_orders.csv:")
print(orders_df.to_string(index=False))

dyson_products.csv:
                      Name  Price     Category                                                 Features
       V15 Detect Absolute    749       Vacuum      Laser dust detection, LCD screen, strongest suction
           V12 Detect Slim    649       Vacuum    Lightweight, laser detection, great for smaller homes
               Cyclone V10    499       Vacuum           Reliable cordless, strong suction, great value
Purifier Cool Formaldehyde    649 Air Purifier              HEPA H13, removes formaldehyde, cooling fan
      Airwrap Multi-Styler    599    Hair Care Curls, dries, styles with Coanda effect, no extreme heat
      Supersonic Hairdryer    429    Hair Care            Fast drying, reduces heat damage, lightweight

dyson_orders.csv:
  Order ID                    Product  Quantity Ordered Order Date Delivery Date     Status
DYS-483921        V15 Detect Absolute                 1 2025-03-01    2025-03-08    Shipped
DYS-291047       Airwrap Multi-Styler            

## Step 3 — Tool Definitions

In [83]:
# ── Step 3: Define Agent Tools ────────────────────────────────────────────────
# The @tool decorator turns a Python function into a LangChain tool that the
# LLM can invoke as part of a ReAct (Reason → Act → Observe) loop.
import pandas as pd
from langchain_core.tools import tool

# Load CSVs into module-level dataframes.
# product_orders_df is mutable so update_quantity() writes back to the same object.
product_pricing_df = pd.read_csv("dyson_products.csv")
product_orders_df  = pd.read_csv("dyson_orders.csv")

# ─────────────────────────────────────────────────────────────────────────────
# PRODUCT TOOL
# ─────────────────────────────────────────────────────────────────────────────
@tool
def get_dyson_price(product_name: str) -> int:
    """
    Returns the price of a Dyson product given its name.
    Performs a case-insensitive substring match on the Name column.
    Returns -1 if no matching product is found.
    """
    match_df = product_pricing_df[
        product_pricing_df["Name"].str.contains(product_name, case=False, na=False)
    ]
    if len(match_df) == 0:
        return -1
    return int(match_df["Price"].iloc[0])

# ─────────────────────────────────────────────────────────────────────────────
# ORDER TOOLS
# ─────────────────────────────────────────────────────────────────────────────
@tool
def get_order_details(order_id: str) -> str:
    """
    Returns the full details of a Dyson order given its order ID (e.g. DYS-483921).
    Performs an exact match on the Order ID column.
    Returns -1 if the order ID is not found.
    """
    match_df = product_orders_df[product_orders_df["Order ID"] == order_id]
    if len(match_df) == 0:
        return -1
    return match_df.iloc[0].to_dict()

@tool
def update_quantity(order_id: str, new_quantity: int) -> bool:
    """
    Updates the ordered quantity for an existing Dyson order.
    Returns True on success, -1 if the order ID is not found.
    """
    match_df = product_orders_df[product_orders_df["Order ID"] == order_id]
    if len(match_df) == 0:
        return -1
    # Update quantity in-place on the module-level dataframe
    product_orders_df.loc[
        product_orders_df["Order ID"] == order_id, "Quantity Ordered"
    ] = new_quantity
    return True

## Step 4 — Product Agent

In [84]:
# ── Step 4: Product Agent ─────────────────────────────────────────────────────
# Uses LangGraph's create_react_agent to build a ReAct loop automatically.
# ReAct = Reason → Act (call tool) → Observe (tool result) → Reason again
# Based on code_03_XX_Product_QnA_Agentic_chatbot.ipynb
# RAG / Chroma / PyPDF removed — only the CSV price tool is used.

from langchain.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import SystemMessage

# System prompt defines the Product Agent's persona and scope
product_system_prompt = """
You are Alex, a Dyson product specialist.
Answer questions about Dyson products. Use the available tools for pricing.
For feature questions and recommendations, use your knowledge of Dyson products.
You can lookup information if needed, but keep it stricty to dyson products.
Do not answer questions unrelated to Dyson products.
"""

# MemorySaver stores conversation history keyed by thread_id
product_memory = MemorySaver()

# create_react_agent builds the ReAct loop: LLM → tool call → observe → LLM
product_QnA_agent = create_agent(
    model=model,
    tools=[get_dyson_price],
    system_prompt=product_system_prompt,
    checkpointer=product_memory
)

## Step 5 — Orders Agent

In [85]:
# ── Step 5: Orders Agent ──────────────────────────────────────────────────────
# Custom agent using a manually built StateGraph (the OrdersAgent class pattern).
# Based on code_04_XX_Orders_Chatbot_with_custom_agent.ipynb
# Graph: orders_llm → (tool call?) → orders_tools → orders_llm (loop until done)

from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict, Annotated
import operator
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage

# Messages are appended (not replaced) across graph steps via operator.add
class OrdersAgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]


class OrdersAgent:
    """Custom ReAct agent for Dyson order queries. Manually builds the LangGraph StateGraph."""

    def __init__(self, model, tools, system_prompt, debug=False):
        self.system_prompt = system_prompt
        self.debug = debug

        # Build the ReAct graph: LLM node → conditional branch → tools node → back to LLM
        agent_graph = StateGraph(OrdersAgentState)
        agent_graph.add_node("orders_llm",   self.call_llm)
        agent_graph.add_node("orders_tools", self.call_tools)
        agent_graph.add_conditional_edges(
            "orders_llm",
            self.is_tool_call,
            {True: "orders_tools", False: END}
        )
        agent_graph.add_edge("orders_tools", "orders_llm")  # tool results feed back into LLM
        agent_graph.set_entry_point("orders_llm")

        # MemorySaver enables multi-turn memory (e.g. recall an order ID across turns)
        self.memory = MemorySaver()
        self.agent_graph = agent_graph.compile(checkpointer=self.memory)

        # Map tool name → tool object for fast dispatch in call_tools()
        self.tools = {t.name: t for t in tools}
        # Bind tools to the model so the LLM knows which tools are available
        self.model = model.bind_tools(tools)

    def call_llm(self, state: OrdersAgentState):
        """Invoke the LLM with the current message history plus the system prompt."""
        messages = state["messages"]
        if self.system_prompt:
            # Prepend system prompt on every call so the persona is always active
            messages = [SystemMessage(content=self.system_prompt)] + messages
        result = self.model.invoke(messages)
        if self.debug:
            print(f"LLM returned: {result}")
        return {"messages": [result]}

    def is_tool_call(self, state: OrdersAgentState):
        """Routing function: returns True if the LLM's last message contains a tool call."""
        last_message = state["messages"][-1]
        return len(last_message.tool_calls) > 0

    def call_tools(self, state: OrdersAgentState):
        """Execute all tool calls requested by the LLM and return ToolMessage results."""
        tool_calls = state["messages"][-1].tool_calls
        results = []
        for tc in tool_calls:
            # Guard against hallucinated tool names
            if tc["name"] not in self.tools:
                result_content = "Unknown tool requested. Please try again."
            else:
                result_content = self.tools[tc["name"]].invoke(tc["args"])
            # Wrap each result in ToolMessage so the LLM can read it on the next step
            results.append(ToolMessage(
                tool_call_id=tc["id"],
                name=tc["name"],
                content=str(result_content)
            ))
        if self.debug:
            print(f"Tools returned: {results}")
        return {"messages": results}


orders_system_prompt = """
You are Alex, a Dyson order management specialist.
You help customers check order status and update order quantities using the available tools.
Always ask for an order ID (format: DYS-XXXXXX) before looking up any order.
Do not reveal details of other customers' orders.
Be concise, professional, and helpful.
"""

orders_agent = OrdersAgent(
    model=model,
    tools=[get_order_details, update_quantity],
    system_prompt=orders_system_prompt,
    debug=False
)

## Step 6 — Refund Agent

In [86]:
# ── Step 6: Refund Agent ──────────────────────────────────────────────────────
# Follows the same custom StateGraph pattern as OrdersAgent but with no tools.
# The full Dyson return/refund policy lives in the system prompt, so no CSV
# or external data source is needed.
# Graph is simpler: refund_llm → END (no conditional tool loop)

# Reuse the same state structure as OrdersAgentState
class RefundAgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]


class RefundAgent:
    """Handles Dyson return/refund policy questions. Policy is static — no tools needed."""

    def __init__(self, model, system_prompt, debug=False):
        self.system_prompt = system_prompt
        self.model = model  # No tools bound — policy is fully in the system prompt
        self.debug = debug

        # Simple linear graph: LLM → END (no tool calls, no conditional routing)
        agent_graph = StateGraph(RefundAgentState)
        agent_graph.add_node("refund_llm", self.call_llm)
        agent_graph.add_edge("refund_llm", END)
        agent_graph.set_entry_point("refund_llm")

        # MemorySaver allows multi-turn refund conversations
        self.memory = MemorySaver()
        self.agent_graph = agent_graph.compile(checkpointer=self.memory)

    def call_llm(self, state: RefundAgentState):
        """Invoke the LLM with the system prompt (containing the policy) and message history."""
        messages = state["messages"]
        if self.system_prompt:
            messages = [SystemMessage(content=self.system_prompt)] + messages
        result = self.model.invoke(messages)
        if self.debug:
            print(f"LLM returned: {result}")
        return {"messages": [result]}


# System prompt embeds the complete Dyson refund policy
refund_system_prompt = """
You are Alex, a Dyson returns and refunds specialist.
Dyson return and refund policy:
- 30-day return window from the date of purchase
- Items must be returned in original condition with all original packaging
- Refunds are processed within 5-10 business days after Dyson receives the return
- Gift purchases extend the return window to 60 days
- Damaged or defective products may qualify for immediate replacement
- To start a return, customers can call 1-866-693-9766
Be concise, professional, and helpful.
"""

refund_agent = RefundAgent(
    model=model,
    system_prompt=refund_system_prompt,
    debug=False
)

## Step 7 — Router & Multi-Agent Assembly

In [87]:
# ── Step 7: Router & Multi-Agent Assembly ─────────────────────────────────────
# Wires all four agents together using a top-level LangGraph StateGraph.
# Based on code_06_XX_Multi-agent_chatbots_with_routing.ipynb

import functools
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, END
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, AIMessage
import operator

# ─────────────────────────────────────────────────────────────────────────────
# agent_node helper (from code_06)
# ─────────────────────────────────────────────────────────────────────────────
def agent_node(agent, name, state, config=None):
    """
    Wraps a compiled sub-agent as a LangGraph graph node.

    The thread_id from the outer graph's config is forwarded into each sub-agent
    so all sub-agents share the same MemorySaver thread. This gives cross-turn
    memory: e.g. an order ID mentioned in turn N is still known in turn N+1.

    'agent' and 'name' are pre-filled by functools.partial; LangGraph injects
    'state' and 'config' at call time.
    """
    if config and isinstance(config, dict):
        thread_id = config.get("configurable", {}).get("thread_id", "shared_thread")
    else:
        thread_id = "shared_thread"

    agent_config = {"configurable": {"thread_id": thread_id}}
    result = agent.invoke(state, agent_config)

    # Extract content from the last message — handle both string and list responses
    content = result["messages"][-1].content
    if isinstance(content, list):
        content = " ".join([c if isinstance(c, str) else str(c) for c in content])

    # Wrap as AIMessage for the shared router state
    return {"messages": [AIMessage(content)]}


# functools.partial pre-fills 'agent' and 'name'; the resulting callables accept
# (state, config) which is what LangGraph passes to each node at runtime
product_QnA_node = functools.partial(agent_node, product_QnA_agent,        "Product_QnA_Agent")
orders_node      = functools.partial(agent_node, orders_agent.agent_graph, "Orders_Agent")
refund_node      = functools.partial(agent_node, refund_agent.agent_graph, "Refund_Agent")

# ─────────────────────────────────────────────────────────────────────────────
# Shared state for the top-level router graph
# ─────────────────────────────────────────────────────────────────────────────
class RouterAgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]


# ─────────────────────────────────────────────────────────────────────────────
# RouterAgent class (from code_06)
# ─────────────────────────────────────────────────────────────────────────────
class RouterAgent:
    """
    Top-level router that classifies every user message and dispatches it
    to the correct specialist agent: Product, Orders, Refund, or SmallTalk.
    """

    def __init__(self, model, system_prompt, smalltalk_prompt, debug=False):
        self.system_prompt    = system_prompt
        self.smalltalk_prompt = smalltalk_prompt
        self.model            = model
        self.debug            = debug

        router_graph = StateGraph(RouterAgentState)
        router_graph.add_node("Router",        self.call_llm)
        router_graph.add_node("Product_Agent", product_QnA_node)
        router_graph.add_node("Orders_Agent",  orders_node)
        router_graph.add_node("Refund_Agent",  refund_node)
        router_graph.add_node("Small_Talk",    self.respond_smalltalk)

        # Conditional routing: the Router's output keyword selects the next node
        router_graph.add_conditional_edges(
            "Router",
            self.find_route,
            {
                "PRODUCT":   "Product_Agent",
                "ORDER":     "Orders_Agent",
                "REFUND":    "Refund_Agent",
                "SMALLTALK": "Small_Talk",
                "END":       END
            }
        )

        # Each specialist node terminates the turn after responding
        router_graph.add_edge("Product_Agent", END)
        router_graph.add_edge("Orders_Agent",  END)
        router_graph.add_edge("Refund_Agent",  END)
        router_graph.add_edge("Small_Talk",    END)

        router_graph.set_entry_point("Router")

        # Sub-agents carry their own MemorySaver; the outer graph needs no checkpointer
        self.router_graph = router_graph.compile()

    def call_llm(self, state: RouterAgentState):
        """Classify the user input and return exactly one routing keyword."""
        messages = state["messages"]
        if self.debug:
            print(f"Router input: {messages[-1].content}")

        context_messages = messages [-2:] if len(messages) >= 2 else messages

        if self.system_prompt:
            context_messages = [SystemMessage(content=self.system_prompt)] + context_messages

        result = self.model.invoke(context_messages)


        if self.system_prompt:
            messages = [SystemMessage(content=self.system_prompt)] + messages
        result = self.model.invoke(messages)
        if self.debug:
            print(f"Router classified as: {result.content}")
        return {"messages": [result]}

    def respond_smalltalk(self, state: RouterAgentState):
        """Handle greetings and goodbyes with a professional on-brand response."""
        messages = state["messages"]
        messages = [SystemMessage(content=self.smalltalk_prompt)] + messages
        result = self.model.invoke(messages)
        return {"messages": [result]}

    def find_route(self, state: RouterAgentState):
        """
        Extract and normalise the routing keyword from the Router's last message.
        Falls back to END for any unrecognised output.
        """
        last_message = state["messages"][-1]
        # Normalise: strip whitespace, take first word, drop trailing punctuation
        raw = last_message.content.strip().upper()
        destination = raw.split()[0].rstrip(".,!?") if raw.split() else "END"
        if self.debug:
            print(f"Routing to: {destination}")

        valid_routes = {"PRODUCT", "ORDER", "REFUND", "SMALLTALK", "END"}

        # Exact match first
        if destination in valid_routes:
            return destination

        # Handle truncation (e.g. "SMALLTAL" → "SMALLTALK")
        for route in valid_routes:
            if route.startswith(destination):
                return route

        return "END"


# ─────────────────────────────────────────────────────────────────────────────
# System prompts
# ─────────────────────────────────────────────────────────────────────────────

# Router prompt: the LLM must output exactly one of the five keywords

router_system_prompt = """
You are a Router that analyzes customer input and chooses one of 5 options:
PRODUCT: If the query is about Dyson products, features, pricing, recommendations, comparisons, or is a follow-up to a product question.
ORDER: If the query is about order status, order details, or updating an order.
REFUND: If the query is about returns, refunds, return policy, or is a follow-up to a refund question.
SMALLTALK: If the input is small talk like greetings or goodbyes.
END: Default, when the input does not match any of the above.

When in doubt between PRODUCT, ORDER, or REFUND and END, always choose the most relevant category.
Never return END for a question that could reasonably be about products, orders, or refunds.

Output only one word: PRODUCT, ORDER, REFUND, SMALLTALK, or END.
"""

smalltalk_prompt = """
You are Alex, a Dyson customer service assistant.
Respond professionally to greetings and goodbyes.
Let the customer know you can help with product questions, order status, and return/refund policy.
"""

router_agent = RouterAgent(
    model=model,
    system_prompt=router_system_prompt,
    smalltalk_prompt=smalltalk_prompt,
    debug=False
)

print("Router assembled. Agents: Product | Orders | Refund | SmallTalk")

Router assembled. Agents: Product | Orders | Refund | SmallTalk


In [88]:
import uuid
from langchain_core.messages import HumanMessage

demo_config = {"configurable": {"thread_id": "demo-" + str(uuid.uuid4())}}

def chat(user_input):
    result = router_agent.router_graph.invoke(
        {"messages": [HumanMessage(user_input)]},
        config=demo_config
    )
    print(f"USER : {user_input}")
    print(f"AGENT: {result['messages'][-1].content}")
    print("-" * 55)

# Turn 1: Smalltalk
chat("Hey, I need some help with a few things today.")

# Turn 2: Product — pricing tool call
chat("I'm trying to decide between the V15 Detect Absolute and the V12 Detect Slim. How much does each cost?")

# Turn 3: Product — memory test, no product re-stated
chat("What are the key differences between those two?")

# Turn 4: Product — follow up still on same topic
chat("Which one would you recommend for a one-bedroom apartment?")

# Turn 5: Refund — pivot to policy
chat("If I buy one and don't like it, what's my return window?")

# Turn 6: Refund — memory test follow up
chat("What if it was a gift? Does that change anything?")

# Turn 7: Order — tool call lookup
chat("Ok great. I also have an existing order I want to check on. Order DYS-667234.")

# Turn 8: Order — memory test, no order ID repeated
chat("Can you update the quantity on that to 1?")

# Turn 9: Order — another order, different ID
chat("Actually can you also check order DYS-104839?")

# Turn 10: Smalltalk — close out
chat("Perfect, that covers everything. Thanks for your help!")

USER : Hey, I need some help with a few things today.
AGENT: Hello! I’m here to help you. What questions do you have today regarding our products, order status, or our return/refund policy?
-------------------------------------------------------
USER : I'm trying to decide between the V15 Detect Absolute and the V12 Detect Slim. How much does each cost?
AGENT: The Dyson V15 Detect Absolute costs $749, while the V12 Detect Slim is priced at $649.
-------------------------------------------------------
USER : What are the key differences between those two?
AGENT: The key differences between the Dyson V15 Detect Absolute and the V12 Detect Slim include:

1. **Suction Power**: The V15 Detect Absolute has more powerful suction than the V12 Detect Slim, making it more effective for deep cleaning.

2. **Laser Technology**: The V15 features advanced laser dust detection technology that reveals microscopic dust on hard floors, whereas the V12 does not have this feature.

3. **Price**: The V15 D